Genomic prediction of Inflixmab in Inflammatory Bowel Disease Patients
This notebook uses baseline mucosal transcriptomic data to predict response to anti-TNF therapy in patients with active IBD refractory to corticosteroids and /or immunosuppression

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import os
import urllib.request
import gzip

# Define the URL for the GSE16879 Series Matrix file from NCBI GEO
data_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz"

# Define the local path where we want to save the file
dest_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# Check if the file already exists locally so we don't re-download it every time
if not os.path.exists(dest_path):
    print("Downloading dataset from NCBI GEO... This might take a minute.")
    urllib.request.urlretrieve(data_url, dest_path)
    print(f"Download complete! File saved to: {dest_path}")
else:
    print("Dataset already exists locally in the 'data' folder.")

In [ ]:
file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# 'rt' means read text. Add explicit encoding and tell it to ignore decoding errors
with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for i in range(40):
        line = f.readline().strip()
        # Show the first 120 characters of each line 
        print(f"Line {i+1}: {line[:120]}")

In this section, we extract patient sample characteristics from the raw NCBI GEO metadata. Healthy controls are removed to isolate IBD (Chron's Diseaseand Ulcerative Colitis) patients and clean the strings to esetablish a Yes / No treatment response

In [ ]:


file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # Stop reading when we hit the numeric data
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break 
                
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            
            # 1. Clean the key (e.g., "title")
            base_key = parts[0].replace('!Sample_', '').strip()
            
            # 2. Clean the values (the patient data list)
            values = [v.replace('"', '').strip() for v in parts[1:]]
            final_key = base_key # Start with the original name (e.g., "characteristics_ch1")
            
            counter = 1          # Start our number tag at 1

            # WHILE the name is already taken in the dictionary...
            while final_key in metadata_dict:
                # create a new name by gluing the counter to the original name
                final_key = f"{base_key}_{counter}"
                # increase the counter by 1 for the next loop (if needed)
                counter +=1
            # ==========================================
            
            # 3. Save the data into the dictionary
            metadata_dict[final_key] = values

# Convert to DataFrame and Transpose
df_clinical = pd.DataFrame.from_dict(metadata_dict).T
list(df_clinical.columns)

In [ ]:
# 1. Transpose 
df_clinical = df_clinical.T

# 2. Select the four characteristics columns, check top 5 rows
df_clinical[['characteristics_ch1', 'characteristics_ch1_1', 'characteristics_ch1_2', 'characteristics_ch1_3']].head(5)

In [ ]:
#removing the controls from the IBD dataframe
df_ibd = df_clinical[ df_clinical['characteristics_ch1_2'] != "response to infliximab: Not applicable"]

#checking controls were removed
df_ibd['characteristics_ch1_2'].head(5)

In [ ]:
#renaming columns
df_ibd = df_ibd.rename(columns={'characteristics_ch1' : 'tissue'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_1' : 'disease'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_2' : 'response to infliximab'})
df_ibd = df_ibd.rename(columns={'characteristics_ch1_3' : 'before or after first infliximab treatment'})

In [ ]:
#remove unnecessary text
df_ibd['tissue'] = df_ibd['tissue'].str.replace("tissue: ", "")
df_ibd['disease'] = df_ibd['disease'].str.replace("disease: ", "")
df_ibd['response to infliximab'] = df_ibd['response to infliximab'].str.replace("response to infliximab: ", "")
df_ibd['before or after first infliximab treatment'] = df_ibd['before or after first infliximab treatment'].str.replace("before or after first infliximab treatment: ", "")

df_ibd[['tissue', 'disease', 'response to infliximab', 'before or after first infliximab treatment']].head(5)

We load the matrix containing 54,675 genes and filter it to only include rows corresponding to IBD patients. To ensure an unbiased evaluation, 20% of data is paritioned into a "hidden" test set before training the model

In [ ]:
df_genes = pd.read_csv(
    file_path , #file defined earler
    sep='\t',   # columns are separated by tabs
    compression=  'gzip', #unzips file
    comment = '!', #ignores clinical header
    index_col = 0 #Gene IDs become row names
)

In [ ]:
# 1. Sets row names to be patient IDs, not numbers
df_ibd = df_ibd.set_index('geo_accession')

# 2. Reload the gene matrix 
df_genes = pd.read_csv(
    file_path, 
    sep='\t', 
    compression='gzip', 
    comment='!', 
    index_col=0
)

# 3. Transpose the gene matrix
df_genes = df_genes.T 

# 4. Filter the gene matrix - only include rows that match patient IDs
df_genes = df_genes.loc[df_ibd.index]

# 5. Check dimensions
print("Clinical shape:", df_ibd.shape)
print("Gene shape:", df_genes.shape)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df_genes, #genes
    df_ibd['response to infliximab'], #what we are trying to predict
    test_size = 0.2, #20% hidden from model
    random_state = 2 #random shuffle is the same each time
)
print(x_train.shape) #output is the 54675 genes for 96 patients (80%), used to train the model
print(x_test.shape) #output is the 54675 genes for 25 patients (20%) that is hidden from the model, "final exam"
#y train = answers for x_train (whether patient reponds) for model training
#y test = answers for x_test (whether patient responds), used to evaulate model performance

We implement a L1-penalised (Lasso) Logistic regression model. The lasso penalty reduces the weights of non-predictive genes to 0, selecting for the most predictive biomarkers 

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
   penalty = 'l1',        #uses lasso penalty - deletes genes that have no impact
   solver = 'liblinear',  # math engine needed for L1 penalty
   random_state = 2 #keeps randomisation consistent
)

model.fit(x_train, y_train) #trains model based on patient genes and response to infliximab (80% of data)
print("Training data analysis finished")

In [ ]:
predictions = model.predict(x_test) #model predicts responses from training data for remainining 20% of patients

from sklearn.metrics import accuracy_score
final_grade = accuracy_score(y_test, predictions) #y_test is patient responses from x_test data, compapres against model predictions

print(f"The model scored: {final_grade * 100: .2f}%") # prints prediction

We pull the non-zero coefficients from the model to identify the genes that survived the lasso filter. These are then sorted to isolate the top 10 most influential biomarkers

In [ ]:
df_results = pd.DataFrame({ #creates dataframe using geness and weights from model
    'genes' : x_train.columns,
    'weight' : model.coef_[0],
})
 
df_results = df_results[df_results['weight'] != 0.0] #keeps weights not equal to 0

print(f"Total surviving genes: {len(df_results)}")             #len counts number of rows in dataframe

In [ ]:
top_10_genes = df_results.sort_values('weight', ascending = False).head(10) # sorts by top 10 genes with most impact, saves genes to variable

top_10_genes.to_csv("infliximab_top_10_biomarkers.csv", index = False) #exports it to folder, index = false stops row numbers being added to file

print("Exported successfully")

import joblib

joblib.dump(model, "infliximab_lasso_model.pkl")